# Explainable-Hybrid-ASAG - full neural + hybrid run (Colab / GPU)

Runs the project's **deferred torch slice** end-to-end and writes every artifact the
manuscript's `\TBD{}` / `TODO(after Colab run)` slots need - turning the research from
*theoretical* into *applied*:

1. **Phase 2G** - standalone DeBERTa-v3 cross-encoder (CORAL/CORN ordinal heads, LoRA) vs the GBM head -> `reports/phase2g/`.
2. **Phase 4 hybrid** - out-of-fold neural signals `neural_oof.parquet` per dataset (row-aligned, leakage-free).
3. **Downstream (CPU, fast)** - `ablations` (branch **-D / only-D**), `neural_compare` (three-way, Table 3), `train2d` (hybrid tuned + significance), `xai` (SHAP now attributes to `neural_*`).
4. **LLM zero-shot baseline** - the 2026 reviewer expectation -> `reports/phase_llm/`.

### Before you run
- **Runtime -> Change runtime type -> GPU** (a free **T4** works; L4 / A100 are much faster).
- Your project zip must be at **`/content/drive/MyDrive/Explainable-Hybrid-ASAG.zip`** and must contain `data/processed/<dataset>/{encoder,features}.parquet`.
- Run **top to bottom**. A short **smoke test** (cell 6) fine-tunes on a tiny slice first, so any environment problem shows up in ~3 minutes, not 3 hours. Stages 2G and OOF are **resumable** - a disconnect only re-does unfinished datasets.
- Rough GPU time (6 datasets, LoRA, 3 seeds): **~2-3 h on A100/L4, ~5-8 h on a free T4**. To sanity-check first, set `DATASETS = ['mohler']` in the driver cell, then scale up.

> After the run finishes, `results_bundle.zip` is downloaded **and** copied to `MyDrive/Explainable-Hybrid-ASAG-results.zip`. Unzip it over your local repo, run `python paper/verify_numbers.py`, and fill the `\TBD{}` slots. The paper is then Q2-ready pending the writing-pass items in the runbook.

In [ ]:
# 1) Dependencies for the neural + hybrid + XAI + LLM slices.
#    Colab ships a MATCHED torch + transformers + numpy (currently Python 3.12 / torch ~2.12,
#    CUDA build). We deliberately DO NOT reinstall or pin torch/transformers/numpy - reinstalling
#    torch is the #1 cause of CUDA breakage, and the model code is written to tolerate new
#    transformers (it forces the DeBERTa head to fp32). We only ADD what the project needs on top:
#      peft                 - LoRA (trains <1% of params; protects Mohler ~616 / SAF ~1.7k rows)
#      sentence-transformers - the SBERT encoder the Phase 2F XAI stage loads (else `xai` crashes)
#      sentencepiece+protobuf - build the DeBERTa-v3 fast tokenizer (it ships no tokenizer.json)
#      accelerate           - transformers training utils
#      spacy + en_core_web_sm - the NER pipe the Phase 2F concept/SAF attribution (xai) needs
#      lightgbm/optuna      - pinned to the paper's versions so the GBM re-runs reproduce numbers
#      loguru               - the project's logger
%pip -q install loguru peft sentence-transformers sentencepiece protobuf accelerate spacy "lightgbm==4.5.0" "optuna==4.1.0"
!python -m spacy download en_core_web_sm
# Colab preinstalls an OLD torchao (0.10) that the fresh peft (0.19) hard-rejects during LoRA
# build (ImportError). peft's LoRA does NOT need torchao -> remove the stale copy so the check
# returns cleanly. Runs before `import peft` (cell 5), so no restart is needed on a fresh run.
!pip -q uninstall -y torchao
print("deps installed - torch/transformers/numpy left exactly as Colab shipped them.")
print("Any pip 'dependency resolver' warning about pre-installed Colab packages is safe here:")
print("nothing above pins torch/transformers, and we don't use the packages those warnings mention.")

In [ ]:
# 2) Mount Google Drive, find your project zip, and extract it.
#    Robust to how the zip was made: it works whether the archive nests the project in a
#    top-level folder or keeps files at the root. We skip only venv/.git/__pycache__ (huge and
#    useless on Colab) but KEEP data/raw if present, so the ASAP-SAS human-ceiling recomputes.
import os, sys, glob, subprocess, shutil
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/Explainable-Hybrid-ASAG.zip'   # <- the path you specified
assert os.path.exists(ZIP), (
    'Project zip not found at ' + ZIP + '. Put your zip in My Drive with exactly that name, '
    'or edit ZIP above to match where it is.')

EXTRACT = '/content/extracted'
if not glob.glob(EXTRACT + '/**/pyproject.toml', recursive=True):     # skip re-extract on re-run
    shutil.rmtree(EXTRACT, ignore_errors=True); os.makedirs(EXTRACT, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', ZIP, '-d', EXTRACT,
                    '-x', '*/venv/*', '*/.git/*', '*/__pycache__/*'], check=False)
cands = [os.path.dirname(p) for p in glob.glob(EXTRACT + '/**/pyproject.toml', recursive=True)]
REPO = next((d for d in cands if os.path.exists(os.path.join(d, 'configs', 'data.yaml'))), None)
assert REPO, ('Could not find the repo root (a folder with BOTH pyproject.toml and '
              'configs/data.yaml) under ' + EXTRACT + '; candidates=' + str(cands))
print('REPO =', REPO)

In [ ]:
# 3) Put src/ on the import path and point the config at the repo. We do NOT `pip install`
#    the project (requires-python is <3.12 and the Arabic source path breaks editable installs) -
#    importing from src/ is enough. Then confirm the processed parquets the run needs are present.
os.environ['ASAG_PROJECT_ROOT'] = REPO
os.environ['PYTHONPATH'] = os.path.join(REPO, 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'src'))

proc = os.path.join(REPO, 'data', 'processed')
need = ('encoder.parquet', 'features.parquet')
present = sorted(d for d in os.listdir(proc)
                 if os.path.isdir(os.path.join(proc, d))
                 and all(os.path.exists(os.path.join(proc, d, f)) for f in need)) if os.path.isdir(proc) else []
assert present, ('No processed parquets found under data/processed/<dataset>/. Your zip must '
                 'include data/processed/<name>/{encoder,features}.parquet (~4.2 MB total). '
                 'Re-zip the project WITH the data/processed folder and try again.')
print('datasets ready (encoder + features present):', present)

In [ ]:
# 4) Write a GPU neural block into configs/data.yaml so every `python -m ...` subprocess below
#    reads a correct, reproducible config (not an in-memory tweak that subprocesses wouldn't see).
#    Idempotent: re-running just resets the block. batch_size auto-scales to the detected GPU.
#    Only keys that exist on NeuralCfg are written (the loader forbids unknown keys).
import yaml, torch
cfg_path = os.path.join(REPO, 'configs', 'data.yaml')
doc = yaml.safe_load(open(cfg_path, encoding='utf-8'))
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
bs = 32 if any(k in gpu for k in ('A100', 'H100', 'L4', 'L40', 'A10', 'V100')) else 16   # free T4 -> 16
doc['neural'] = {
    'enabled': True, 'backbone': 'microsoft/deberta-v3-base', 'device': 'cuda',
    'epochs': 4, 'batch_size': bs, 'lr': 2.0e-5, 'warmup_ratio': 0.1, 'dropout': 0.1,
    'ordinal_head': 'corn', 'pooling': 'cls',
    'seeds': [42, 1, 2],                    # 3 seeds -> publication-grade mean +/- std
    'num_workers': 0, 'select_on_dev': True, 'save_predictions': True,
    'lora_enabled': True, 'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1,
    'max_len': 128, 'max_len_overrides': {'saf': 256},
}
yaml.safe_dump(doc, open(cfg_path, 'w', encoding='utf-8'), sort_keys=False, allow_unicode=True)
print('GPU:', gpu, '| batch_size=' + str(bs), '| seeds=[42,1,2] | LoRA on | epochs=4 | ordinal=corn')

In [ ]:
# 5) Fail-fast: verify the exact libraries + APIs the run needs resolve on THIS runtime, and that
#    the config reloads with the GPU neural block, BEFORE anything expensive.
import transformers, peft, lightgbm, optuna, sentence_transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup   # stable 4.x -> 5.x
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, '| peft', peft.__version__,
      '| sentence-transformers', sentence_transformers.__version__,
      '| lightgbm', lightgbm.__version__, '| optuna', optuna.__version__)
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU (T4 is fine), then re-run from cell 1.'
# deberta-v3 ships no tokenizer.json -> the fast tokenizer is built from SentencePiece via protobuf.
_tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
_enc = _tok('a ref [SEP] b', 'student', truncation=True, max_length=16, padding='max_length', return_tensors='pt')
assert _enc['input_ids'].shape[1] == 16, 'tokenizer sanity failed'
import asag
from asag.config import load_data_config
load_data_config.cache_clear()                       # drop any cached config from before the patch
cfg = load_data_config()
assert cfg.neural.device == 'cuda' and cfg.neural.lora_enabled, 'neural config not applied - re-run cell 4'
print('asag OK from', asag.__file__, '| neural.seeds =', cfg.neural.seeds)
print('OK - libraries, tokenizer, and config all resolve on this runtime')

In [ ]:
# 6) SMOKE TEST (~2-3 min, one-time). Actually fine-tune DeBERTa+LoRA on a tiny slice for each
#    task family (regression / ordinal / classification) and load the SBERT encoder the XAI stage
#    uses. This exercises the FULL torch+transformers+peft+deberta path AND downloads/caches the
#    model, so any incompatibility surfaces now - not hours into the real run.
import numpy as np, pandas as pd
from asag.models.tasks import get_spec
from asag.neural.evaluate_neural import load_text_df
from asag.neural.trainer import fit_predict

def _tiny(df, per_group=8, n=28):
    # small slice with >=2 target values in train (LabelSpace needs >=2 for ordinal/classification)
    key = (df['label'].astype(str) if ('label' in df.columns and df['label'].notna().any())
           else pd.to_numeric(df['score'], errors='coerce').round())
    s = pd.concat([g.head(per_group) for _, g in df.groupby(key)]).head(n)
    s = s.sample(frac=1.0, random_state=42).reset_index(drop=True)
    cut = max(4, int(len(s) * 0.7))
    return s.iloc[:cut].reset_index(drop=True), s.iloc[cut:].reset_index(drop=True)

smoke_cfg = cfg.neural.model_copy(update={'epochs': 1, 'batch_size': 8, 'num_workers': 0, 'select_on_dev': False})
for ds in ('mohler', 'mindreading', 'powergrading'):     # regression / ordinal / classification
    if ds not in present:
        print('  smoke skip (not in zip):', ds); continue
    spec = get_spec(ds)
    tr, te = _tiny(load_text_df(ds, cfg))
    r = fit_predict(tr, te, spec, smoke_cfg, seed=42, max_len=64, tokenizer=_tok)
    ok = np.isfinite(np.asarray(r['y_cont'])).all() and len(r['y_pred']) == len(te)
    print('  smoke ' + ds.ljust(12) + ' [' + spec.task_type.ljust(14) + '] pred[:3]=' +
          str(np.asarray(r['y_pred'])[:3]) + '  ok=' + str(ok))
    assert ok, 'smoke test failed for ' + ds + ' - do NOT start the long run; read the error above'

# SBERT (sentence-transformers) - the Phase 2F XAI stage needs this; verify it loads + embeds.
from asag.features.semantic import SbertEncoder
_sb = SbertEncoder(cfg.features.sbert_model, batch_size=8, normalize=True)
print('  smoke SBERT embed shape =', np.asarray(_sb.embed(['a short answer', 'another answer'])).shape)
# spaCy NER pipe - the Phase 2F concept/SAF attribution needs en_core_web_sm (installed in cell 1).
from asag.features.text_utils import load_feature_nlp
_nlp = load_feature_nlp(cfg.features.ner.spacy_model)
print('  smoke spaCy pipe OK:', _nlp.pipe_names)
print('SMOKE OK - full neural + SBERT + spaCy path works here; safe to launch the long run below')

## Driver
`DATASETS` is ordered small -> large so early progress is visible and a free-T4 disconnect costs
the least. Every stage below **skips work whose output already exists**, so you can re-run this
cell to resume. To smoke the whole pipeline end-to-end first, set `DATASETS = ['mohler']`.

In [ ]:
DATASETS = ['mohler', 'powergrading', 'mindreading', 'saf', 'asap_sas', 'semeval']  # small -> large
DATASETS = [d for d in DATASETS if d in present]
def run(*mod_args, extra_env=None):
    env = dict(os.environ, **(extra_env or {}))
    print('>> python -m', *mod_args, flush=True)
    return subprocess.run([sys.executable, '-m', *mod_args], cwd=REPO, env=env).returncode
print('will process:', DATASETS)

In [ ]:
# 7) Phase 2G (Drive-checkpointed) - standalone DeBERTa vs GBM. Each finished dataset is mirrored to
#    MyDrive/asag_ckpt, so a runtime recycle (which wipes /content) never costs you a completed dataset.
#    On a fresh runtime this cell first RESTORES finished datasets from Drive, then resumes the rest.
import json, glob
from asag.config import ensure_dirs
from asag.neural import NEURAL_SCHEMA_VERSION
from asag.neural.run import _write_figure
ensure_dirs(cfg)
CKPT = '/content/drive/MyDrive/asag_ckpt'
os.makedirs(CKPT + '/phase2g_partial', exist_ok=True)
os.makedirs(CKPT + '/neural_oof', exist_ok=True)
p2g = os.path.join(REPO, 'reports', 'phase2g'); part = os.path.join(p2g, '_partial')
os.makedirs(part, exist_ok=True)

# resume: pull any previously-finished datasets back from Drive into the fresh /content repo
for src in glob.glob(CKPT + '/phase2g_partial/*.json'):
    shutil.copy2(src, os.path.join(part, os.path.basename(src)))
for src in glob.glob(CKPT + '/neural_oof/*.parquet'):
    dd = os.path.join(REPO, 'data', 'processed', os.path.basename(src)[:-8])
    if os.path.isdir(dd):
        shutil.copy2(src, os.path.join(dd, 'neural_oof.parquet'))
print('restored from Drive:', sorted(os.path.basename(p)[:-5] for p in glob.glob(part + '/*.json')))

for ds in DATASETS:
    cache = os.path.join(part, ds + '.json')
    if os.path.exists(cache):
        print('skip (cached):', ds); continue
    if run('asag.neural.run', ds) == 0 and os.path.exists(os.path.join(p2g, 'results.json')):
        block = json.load(open(os.path.join(p2g, 'results.json'))).get('datasets', {}).get(ds)
        if block:
            json.dump(block, open(cache, 'w'), default=str)
            shutil.copy2(cache, CKPT + '/phase2g_partial/' + ds + '.json')   # -> Drive immediately
            print('  checkpointed', ds, '-> Drive')
blocks = {ds: json.load(open(os.path.join(part, ds + '.json')))
          for ds in DATASETS if os.path.exists(os.path.join(part, ds + '.json'))}
if blocks:
    json.dump({'schema_version': NEURAL_SCHEMA_VERSION, 'backbone': cfg.neural.backbone,
               'seeds': list(cfg.neural.seeds), 'datasets': blocks},
              open(os.path.join(p2g, 'results.json'), 'w'), indent=2, default=str)
    _write_figure(cfg, blocks)
    print('Phase 2G complete for:', list(blocks))
else:
    print('Phase 2G produced no result blocks - check the subprocess logs above')

In [ ]:
# 8) Phase 4 hybrid (Drive-checkpointed) - out-of-fold DeBERTa signals -> neural_oof.parquet, each
#    mirrored to Drive as soon as it is produced. THE critical artifact (feeds ablations -D, three-way,
#    train2d, xai). Resumable per dataset (restored from Drive by the Phase 2G cell above).
CKPT = '/content/drive/MyDrive/asag_ckpt'; os.makedirs(CKPT + '/neural_oof', exist_ok=True)
for ds in DATASETS:
    oof = os.path.join(REPO, 'data', 'processed', ds, 'neural_oof.parquet')
    if os.path.exists(oof):
        print('skip (exists):', ds); continue
    if run('asag.neural.extract_features', ds) == 0 and os.path.exists(oof):
        shutil.copy2(oof, CKPT + '/neural_oof/' + ds + '.parquet')   # -> Drive immediately
        print('  checkpointed', ds, 'OOF -> Drive')

In [ ]:
# 9) Verify OOF coverage - every row must receive an out-of-fold prediction (~100%). Anything
#    lower means a fold/split row was left uncovered; investigate before trusting the hybrid.
import pandas as pd, numpy as np
for ds in DATASETS:
    oof = os.path.join(REPO, 'data', 'processed', ds, 'neural_oof.parquet')
    if os.path.exists(oof):
        d = pd.read_parquet(oof)
        print(ds.ljust(14), 'rows=' + str(len(d)).rjust(6),
              ' OOF coverage=' + '{:.1%}'.format(np.isfinite(d['neural_score']).mean()))
    else:
        print(ds.ljust(14), 'MISSING neural_oof.parquet - re-run cell 8 for this dataset')

In [ ]:
# 10) Downstream GBM-with-neural (CPU, fast). load_bundle auto-concatenates the neural_* columns,
#     so every stage below becomes the hybrid. Each runs as a subprocess; a non-zero exit is
#     reported but non-fatal (the neural artifacts above are already saved).
summary = {}
for mod in ('asag.models.ablations',       # branch D: -D / only-D now populated
            'asag.models.neural_compare',  # three-way neural-only / feature-only / hybrid -> reports/phase_hybrid/
            'asag.models.train2d',         # hybrid tuned headline + paired-bootstrap significance
            'asag.xai.run'):               # SHAP now attributes to neural_* (attribution-dilution)
    summary[mod] = run(mod)
print('downstream exit codes:', summary)
bad = [m for m, c in summary.items() if c != 0]
print('OK - all downstream stages succeeded' if not bad
      else 'WARNING: nonzero exit for ' + str(bad) + ' - scroll up for the traceback')

## LLM zero-shot baseline
Grades each dataset's headline split under the **same** metric/protocol as the other arms,
temperature 0, prompts logged. Default model `Qwen/Qwen2.5-3B-Instruct` fits a T4. This is the
LLM row 2026 reviewers expect. It runs last and is **non-fatal** - the neural/hybrid results are
already saved, so a failure here (e.g. VRAM) doesn't cost you the run.

In [ ]:
LLM_BASELINE_B64 = (
    'IiIiWmVyby1zaG90IExMTSBncmFkaW5nIGJhc2VsaW5lIHVuZGVyIHRoZSBwYXBlcidzIG93biBtZXRyaWNzL3Byb3RvY29sLgoKR3JhZGVzIGVhY2ggZGF0YXNldCdzICoqaGVhZGxpbmUgc3BsaXQqKiB3aXRoIGFuIGluc3RydWN0aW9uLXR1bmVkIExMTSAodGVtcGVyYXR1cmUKMCwgcHJvbXB0cyBsb2dnZWQpLCB0aGVuIHNjb3JlcyBpdCB3aXRoIHRoZSAqc2FtZSogbWV0cmljIHRoZSBHQk0vRGVCRVJUYSBhcm1zIHVzZQooYGBhc2FnLm1vZGVscy5tZXRyaWNzYGAgdmlhIGBgYXNhZy5tb2RlbHMudGFza3MuUkVHSVNUUllgYCkuIFRoaXMgaXMgdGhlIExMTSByb3cKcmV2aWV3ZXJzIGV4cGVjdCBpbiAyMDI2OyB0aGUgcHJvamVjdCdzIG93biByZXZpZXcgc2ltdWxhdGlvbiBmbGFncyBpdHMgYWJzZW5jZSBhcyBhCm5lYXItcmVqZWN0IGF0IFEyIChSZXZpZXdlciBDKS4KCkRlc2lnbiBjaG9pY2VzIChhbGwgZGVsaWJlcmF0ZSwgYWxsIGRvY3VtZW50ZWQgc28gYSByZXZpZXdlciBjYW4gY2hlY2spOgoKKiAqKlplcm8tc2hvdCwgbm8gdHJhaW5pbmcsIG5vIGZvbGRzLioqIGBgb2ZmaWNpYWxfc3BsaXRgYCBkYXRhc2V0cyBhcmUgZ3JhZGVkIG9uIHRoZQogIGxhc3QgLyBoYXJkZXN0IHRlc3Qgc3BsaXQgKFNlbUV2YWwgYGB0ZXN0X3VkYGAsIFNBRiBgYHRlc3RfdXFgYCwgQVNBUC1TQVMKICBgYHRlc3RfdWFgYCk7IGBga2ZvbGRgYCBkYXRhc2V0cyAoTW9obGVyLCBQb3dlcmdyYWRpbmcsIE1JTkQtQ0EpIGFyZSBncmFkZWQgb25jZSBvbgogIGFsbCBDViByb3dzIOKAlCB0aGVyZSBpcyBub3RoaW5nIHRvIGZpdCwgc28gdGhlIGZvbGQgc3RydWN0dXJlIGlzIGlycmVsZXZhbnQuCiogKipUYXNrLWF3YXJlIHByb21wdCArIG1ldHJpYy4qKiBjbGFzc2lmaWNhdGlvbiDihpIgcGljayBvbmUgbGFiZWwgKG1hY3JvLUYxIG9uIHNoYXJlZAogIGNvZGVzKTsgb3JkaW5hbCDihpIgYW4gaW50ZWdlciBpbiB0aGUgb2JzZXJ2ZWQgcmFuZ2UgKFFXSyk7IHJlZ3Jlc3Npb24g4oaSIGEgbnVtYmVyIGluCiAgcmFuZ2UgKFBlYXJzb24pLiBQZXItcHJvbXB0IGRhdGFzZXRzIChBU0FQLVNBUykgYXZlcmFnZSB0aGUgbWV0cmljIGFjcm9zcyBwcm9tcHRzLAogIGV4YWN0bHkgbGlrZSB0aGUgR0JNIGhlYWRsaW5lLgoqICoqQ29zdCBjYXAuKiogYGBMTE1fTUFYX1JPV1NgYCAoZGVmYXVsdCA0MDApIHN1YnNhbXBsZXMgbGFyZ2UgdGVzdCBzcGxpdHMgd2l0aCBhCiAgZml4ZWQgc2VlZDsgYGBuX2V2YWxgYCBpcyByZXBvcnRlZCBzbyB0aGUgZXN0aW1hdGUncyBiYXNpcyBpcyBleHBsaWNpdC4KClN3YXAgYGBMTE1fTU9ERUxgYCBmb3IgYSBzdHJvbmdlciBvcGVuIG1vZGVsICg3Qi84Qikgb3Igd2lyZSBhbiBBUEkgaW5zaWRlCmBgZ2VuZXJhdGUoKWBgIGZvciB0aGUgaGVhZGxpbmUgcm93OyB0aGUgbWV0cmljIHBsdW1iaW5nIGlzIHVuY2hhbmdlZC4KCiAgICBweXRob24gZXhwZXJpbWVudHMvbGxtX3plcm9zaG90X2Jhc2VsaW5lLnB5IFtkYXRhc2V0IC4uLl0KICAgIExMTV9NT0RFTD1Rd2VuL1F3ZW4yLjUtM0ItSW5zdHJ1Y3QgTExNX01BWF9ST1dTPTQwMCBweXRob24gZXhwZXJpbWVudHMvbGxtX3plcm9zaG90X2Jhc2VsaW5lLnB5CgpPdXRwdXRzOiByZXBvcnRzL3BoYXNlX2xsbS97bGxtX2Jhc2VsaW5lLmpzb24sIGV4YW1wbGVzLmpzb259LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3lzCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBhc2FnLmNvbmZpZyBpbXBvcnQgbG9hZF9kYXRhX2NvbmZpZwpmcm9tIGFzYWcubW9kZWxzLm1ldHJpY3MgaW1wb3J0IGNvbXB1dGVfbWV0cmljcwpmcm9tIGFzYWcubW9kZWxzLnRhc2tzIGltcG9ydCBSRUdJU1RSWSwgZ2V0X3NwZWMKCk1PREVMID0gb3MuZW52aXJvbi5nZXQoIkxMTV9NT0RFTCIsICJRd2VuL1F3ZW4yLjUtM0ItSW5zdHJ1Y3QiKQpNQVhfUk9XUyA9IGludChvcy5lbnZpcm9uLmdldCgiTExNX01BWF9ST1dTIiwgIjQwMCIpKSAgICAgICAgICAjIGNhcCBldmFsIHJvd3MvZGF0YXNldCAoMCA9IGFsbCkKTUFYX1BFUl9QUk9NUFQgPSBpbnQob3MuZW52aXJvbi5nZXQoIkxMTV9NQVhfUEVSX1BST01QVCIsICIxNTAiKSkKQkFUQ0ggPSBpbnQob3MuZW52aXJvbi5nZXQoIkxMTV9CQVRDSCIsICIxNiIpKQpTRUVEID0gNDIKCgpkZWYgX2V2YWxfcm93cyhkZjogcGQuRGF0YUZyYW1lLCBzcGVjKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJIZWFkbGluZS1zcGxpdCByb3dzIGZvciBvZmZpY2lhbF9zcGxpdDsgYWxsIENWIHJvd3MgZm9yIGtmb2xkLiBDb3N0LWNhcHBlZC4iIiIKICAgIGlmIHNwZWMucHJvdG9jb2wgPT0gImtmb2xkIjoKICAgICAgICBtYXNrID0gcGQudG9fbnVtZXJpYyhkZlsiZm9sZCJdLCBlcnJvcnM9ImNvZXJjZSIpLmZpbGxuYSgtMSkgPj0gMAogICAgZWxzZToKICAgICAgICBtYXNrID0gZGZbInNwbGl0Il0uYXN0eXBlKHN0cikgPT0gc3BlYy50ZXN0X3NwbGl0c1stMV0KICAgIGQgPSBkZlttYXNrXS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICBpZiBzcGVjLnBlcl9wcm9tcHQgYW5kIE1BWF9QRVJfUFJPTVBUOgogICAgICAgIGQgPSAoZC5ncm91cGJ5KCJxdWVzdGlvbl9pZCIsIGdyb3VwX2tleXM9RmFsc2UpCiAgICAgICAgICAgICAgIC5hcHBseShsYW1iZGEgZzogZy5zYW1wbGUobj1taW4obGVuKGcpLCBNQVhfUEVSX1BST01QVCksIHJhbmRvbV9zdGF0ZT1TRUVEKSkKICAgICAgICAgICAgICAgLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkpCiAgICBlbGlmIE1BWF9ST1dTIGFuZCBsZW4oZCkgPiBNQVhfUk9XUzoKICAgICAgICBkID0gZC5zYW1wbGUobj1NQVhfUk9XUywgcmFuZG9tX3N0YXRlPVNFRUQpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIHJldHVybiBkCgoKZGVmIF9sYWJlbF9zcGFjZShkZjogcGQuRGF0YUZyYW1lLCBzcGVjKSAtPiBkaWN0OgogICAgIiIiQWxsb3dlZCBsYWJlbHMgKGNsYXNzaWZpY2F0aW9uKSBvciBudW1lcmljIHJhbmdlIChvcmRpbmFsL3JlZ3Jlc3Npb24pLiIiIgogICAgaWYgc3BlYy50YXNrX3R5cGUgPT0gImNsYXNzaWZpY2F0aW9uIjoKICAgICAgICBsYWJzID0gc29ydGVkKHMgZm9yIHMgaW4gZGZbImxhYmVsIl0uYXN0eXBlKHN0cikudW5pcXVlKCkgaWYgcyBhbmQgcy5sb3dlcigpICE9ICJuYW4iKQogICAgICAgIHJldHVybiB7ImxhYmVscyI6IGxhYnMsICJ2b2NhYiI6IHtsYWI6IGkgZm9yIGksIGxhYiBpbiBlbnVtZXJhdGUobGFicyl9fQogICAgcyA9IHBkLnRvX251bWVyaWMoZGZbInNjb3JlIl0sIGVycm9ycz0iY29lcmNlIikuZHJvcG5hKCkKICAgIHJldHVybiB7ImxvIjogZmxvYXQocy5taW4oKSksICJoaSI6IGZsb2F0KHMubWF4KCkpfQoKCmRlZiBfcHJvbXB0KHJvdzogcGQuU2VyaWVzLCBzcGVjLCBzcGFjZTogZGljdCkgLT4gc3RyOgogICAgcSA9IHN0cihyb3cuZ2V0KCJxdWVzdGlvbl9lbmMiKSBvciAiIikuc3RyaXAoKQogICAgcmVmID0gc3RyKHJvdy5nZXQoInJlZmVyZW5jZV9hbnN3ZXJfZW5jIikgb3IgIiIpLnN0cmlwKCkKICAgIGFucyA9IHN0cihyb3cuZ2V0KCJzdHVkZW50X2Fuc3dlcl9lbmMiKSBvciAiIikuc3RyaXAoKQogICAgY3R4ID0gIiIKICAgIGlmIHEgYW5kIHEubG93ZXIoKSAhPSAibmFuIjoKICAgICAgICBjdHggKz0gZiJRdWVzdGlvbjoge3F9XG4iCiAgICBpZiByZWYgYW5kIHJlZi5sb3dlcigpICE9ICJuYW4iOgogICAgICAgIGN0eCArPSBmIlJlZmVyZW5jZSBhbnN3ZXI6IHtyZWZ9XG4iCiAgICBpZiBzcGVjLnRhc2tfdHlwZSA9PSAiY2xhc3NpZmljYXRpb24iOgogICAgICAgIHRhc2sgPSAoZiJDbGFzc2lmeSB0aGUgc3R1ZGVudCBhbnN3ZXIgaW50byBleGFjdGx5IG9uZSBvZiB0aGVzZSBsYWJlbHM6ICIKICAgICAgICAgICAgICAgIGYieycsICcuam9pbihzcGFjZVsnbGFiZWxzJ10pfS5cblJlcGx5IHdpdGggT05MWSB0aGUgbGFiZWwsIG5vdGhpbmcgZWxzZS4iKQogICAgZWxpZiBzcGVjLnRhc2tfdHlwZSA9PSAib3JkaW5hbCI6CiAgICAgICAgdGFzayA9IChmIkdyYWRlIHRoZSBzdHVkZW50IGFuc3dlciB3aXRoIGFuIElOVEVHRVIgc2NvcmUgZnJvbSB7aW50KHNwYWNlWydsbyddKX0gdG8gIgogICAgICAgICAgICAgICAgZiJ7aW50KHNwYWNlWydoaSddKX0gKGhpZ2hlciA9IGJldHRlcikuIFJlcGx5IHdpdGggT05MWSB0aGUgaW50ZWdlci4iKQogICAgZWxzZToKICAgICAgICB0YXNrID0gKGYiR3JhZGUgdGhlIHN0dWRlbnQgYW5zd2VyIHdpdGggYSBzY29yZSBmcm9tIHtzcGFjZVsnbG8nXTouMmZ9IHRvICIKICAgICAgICAgICAgICAgIGYie3NwYWNlWydoaSddOi4yZn0gKGhpZ2hlciA9IGJldHRlcikuIFJlcGx5IHdpdGggT05MWSB0aGUgbnVtYmVyLiIpCiAgICByZXR1cm4gZiJ7Y3R4fVN0dWRlbnQgYW5zd2VyOiB7YW5zfVxuXG57dGFza30iCgoKZGVmIF9wYXJzZSh0ZXh0OiBzdHIsIHNwZWMsIHNwYWNlOiBkaWN0KSAtPiBmbG9hdDoKICAgIHQgPSAodGV4dCBvciAiIikuc3RyaXAoKQogICAgaWYgc3BlYy50YXNrX3R5cGUgPT0gImNsYXNzaWZpY2F0aW9uIjoKICAgICAgICBsb3cgPSB0Lmxvd2VyKCkKICAgICAgICBmb3IgbGFiIGluIHNwYWNlWyJsYWJlbHMiXTogICAgICAgICAgICAgICAgICAgICAgICMgZmlyc3QgbGFiZWwgbWVudGlvbmVkIHdpbnMKICAgICAgICAgICAgaWYgbGFiLmxvd2VyKCkgaW4gbG93OgogICAgICAgICAgICAgICAgcmV0dXJuIGZsb2F0KHNwYWNlWyJ2b2NhYiJdW2xhYl0pCiAgICAgICAgcmV0dXJuIGZsb2F0KHNwYWNlWyJ2b2NhYiJdW3NwYWNlWyJsYWJlbHMiXVswXV0pICAjIGZhbGxiYWNrOiBmaXJzdCBsYWJlbAogICAgbSA9IHJlLnNlYXJjaChyIi0/XGQrKD86XC5cZCspPyIsIHQpCiAgICBtaWQgPSAoc3BhY2VbImxvIl0gKyBzcGFjZVsiaGkiXSkgLyAyLjAKICAgIGlmIG5vdCBtOgogICAgICAgIHJldHVybiByb3VuZChtaWQpIGlmIHNwZWMudGFza190eXBlID09ICJvcmRpbmFsIiBlbHNlIG1pZAogICAgdiA9IG1heChzcGFjZVsibG8iXSwgbWluKHNwYWNlWyJoaSJdLCBmbG9hdChtLmdyb3VwKCkpKSkKICAgIHJldHVybiBmbG9hdChyb3VuZCh2KSkgaWYgc3BlYy50YXNrX3R5cGUgPT0gIm9yZGluYWwiIGVsc2UgdgoKCmRlZiBsb2FkX2xsbSgpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsRm9yQ2F1c2FsTE0sIEF1dG9Ub2tlbml6ZXIKCiAgICB0b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChNT0RFTCkKICAgIGlmIHRvay5wYWRfdG9rZW4gaXMgTm9uZToKICAgICAgICB0b2sucGFkX3Rva2VuID0gdG9rLmVvc190b2tlbgogICAgdG9rLnBhZGRpbmdfc2lkZSA9ICJsZWZ0IiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgY29ycmVjdCBiYXRjaGVkIGRlY29kZXItb25seSBnZW4KICAgIG1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIE1PREVMLCB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgIGRldmljZV9tYXA9ImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1vZGVsLmV2YWwoKQogICAgcmV0dXJuIHRvaywgbW9kZWwKCgpkZWYgZ2VuZXJhdGUodG9rLCBtb2RlbCwgcHJvbXB0czogbGlzdFtzdHJdKSAtPiBsaXN0W3N0cl06CiAgICBpbXBvcnQgdG9yY2gKCiAgICBzeXNfbXNnID0gIllvdSBhcmUgYSBzdHJpY3QgZ3JhZGluZyBhc3Npc3RhbnQuIEZvbGxvdyB0aGUgcmVxdWVzdGVkIG91dHB1dCBmb3JtYXQgZXhhY3RseS4iCiAgICB0ZXh0cyA9IFt0b2suYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICBbeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50Ijogc3lzX21zZ30sIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBwfV0sCiAgICAgICAgdG9rZW5pemU9RmFsc2UsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1UcnVlKSBmb3IgcCBpbiBwcm9tcHRzXQogICAgb3V0OiBsaXN0W3N0cl0gPSBbXQogICAgZm9yIGkgaW4gcmFuZ2UoMCwgbGVuKHRleHRzKSwgQkFUQ0gpOgogICAgICAgIGNodW5rID0gdGV4dHNbaTppICsgQkFUQ0hdCiAgICAgICAgZW5jID0gdG9rKGNodW5rLCByZXR1cm5fdGVuc29ycz0icHQiLCBwYWRkaW5nPVRydWUsIHRydW5jYXRpb249VHJ1ZSwKICAgICAgICAgICAgICAgICAgbWF4X2xlbmd0aD0xMDI0KS50byhtb2RlbC5kZXZpY2UpCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIGdlbiA9IG1vZGVsLmdlbmVyYXRlKCoqZW5jLCBtYXhfbmV3X3Rva2Vucz0xMiwgZG9fc2FtcGxlPUZhbHNlLCAgICMgdGVtcCAwIChncmVlZHkpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHBhZF90b2tlbl9pZD10b2sucGFkX3Rva2VuX2lkKQogICAgICAgIGZvciBqIGluIHJhbmdlKGxlbihjaHVuaykpOgogICAgICAgICAgICBvdXQuYXBwZW5kKHRvay5kZWNvZGUoZ2VuW2pdW2VuY1siaW5wdXRfaWRzIl0uc2hhcGVbMV06XSwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKSkKICAgICAgICBwcmludChmIiAgICBnZW5lcmF0ZWQge21pbihpICsgQkFUQ0gsIGxlbih0ZXh0cykpfS97bGVuKHRleHRzKX0iLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIG91dAoKCmRlZiBzY29yZV9kYXRhc2V0KG5hbWU6IHN0ciwgY2ZnLCB0b2ssIG1vZGVsLCBleGFtcGxlczogZGljdCkgLT4gZGljdDoKICAgIHNwZWMgPSBnZXRfc3BlYyhuYW1lKQogICAgcGF0aCA9IGNmZy5wYXRocy5wcm9jZXNzZWQgLyBuYW1lIC8gImVuY29kZXIucGFycXVldCIKICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOgogICAgICAgIHJldHVybiB7InN0YXR1cyI6ICJub19lbmNvZGVyX3BhcnF1ZXQifQogICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQocGF0aCkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgc3BhY2UgPSBfbGFiZWxfc3BhY2UoZGYsIHNwZWMpCiAgICBkID0gX2V2YWxfcm93cyhkZiwgc3BlYykKICAgIGlmIGQuZW1wdHk6CiAgICAgICAgcmV0dXJuIHsic3RhdHVzIjogIm5vX2V2YWxfcm93cyJ9CgogICAgcHJvbXB0cyA9IFtfcHJvbXB0KHIsIHNwZWMsIHNwYWNlKSBmb3IgXywgciBpbiBkLml0ZXJyb3dzKCldCiAgICByYXcgPSBnZW5lcmF0ZSh0b2ssIG1vZGVsLCBwcm9tcHRzKQogICAgeV9wcmVkID0gbnAuYXJyYXkoW19wYXJzZSh0LCBzcGVjLCBzcGFjZSkgZm9yIHQgaW4gcmF3XSwgZHR5cGU9ZmxvYXQpCiAgICBpZiBzcGVjLnRhc2tfdHlwZSA9PSAiY2xhc3NpZmljYXRpb24iOgogICAgICAgIHlfdHJ1ZSA9IGRbImxhYmVsIl0uYXN0eXBlKHN0cikubWFwKHNwYWNlWyJ2b2NhYiJdKS50b19udW1weShkdHlwZT1mbG9hdCkKICAgIGVsc2U6CiAgICAgICAgeV90cnVlID0gcGQudG9fbnVtZXJpYyhkWyJzY29yZSJdLCBlcnJvcnM9ImNvZXJjZSIpLnRvX251bXB5KGR0eXBlPWZsb2F0KQogICAgZmluID0gbnAuaXNmaW5pdGUoeV90cnVlKSAmIG5wLmlzZmluaXRlKHlfcHJlZCkKICAgIGV4YW1wbGVzW25hbWVdID0gW3sicHJvbXB0IjogcHJvbXB0c1trXSwgImxsbV9yYXciOiByYXdba10sICJwcmVkIjogZmxvYXQoeV9wcmVkW2tdKSwKICAgICAgICAgICAgICAgICAgICAgICAidHJ1ZSI6IGZsb2F0KHlfdHJ1ZVtrXSl9IGZvciBrIGluIHJhbmdlKG1pbigzLCBsZW4ocHJvbXB0cykpKV0KCiAgICBkZWYgbWV0cmljKG1hc2s6IG5wLm5kYXJyYXkpIC0+IGZsb2F0OgogICAgICAgIHJldHVybiBjb21wdXRlX21ldHJpY3MoeV90cnVlW21hc2tdLCB5X3ByZWRbbWFza10sIChzcGVjLmhlYWRsaW5lLCkpLmdldCgKICAgICAgICAgICAgc3BlYy5oZWFkbGluZSwgZmxvYXQoIm5hbiIpKQoKICAgIGlmIHNwZWMucGVyX3Byb21wdDoKICAgICAgICBxaWQgPSBkWyJxdWVzdGlvbl9pZCJdLmFzdHlwZShzdHIpLnRvX251bXB5KCkKICAgICAgICB2YWxzID0gW3YgZm9yIHAgaW4gbnAudW5pcXVlKHFpZFtmaW5dKSBpZiBucC5pc2Zpbml0ZSh2IDo9IG1ldHJpYyhmaW4gJiAocWlkID09IHApKSldCiAgICAgICAgaGVhZGxpbmUgPSBmbG9hdChucC5tZWFuKHZhbHMpKSBpZiB2YWxzIGVsc2UgZmxvYXQoIm5hbiIpCiAgICBlbHNlOgogICAgICAgIGhlYWRsaW5lID0gbWV0cmljKGZpbikKICAgIHJldHVybiB7InN0YXR1cyI6ICJvayIsICJtZXRyaWMiOiBzcGVjLmhlYWRsaW5lLAogICAgICAgICAgICAiaGVhZGxpbmVfc3BsaXQiOiBzcGVjLnRlc3Rfc3BsaXRzWy0xXSBpZiBzcGVjLnByb3RvY29sID09ICJvZmZpY2lhbF9zcGxpdCIgZWxzZSAiY3YiLAogICAgICAgICAgICAibl9ldmFsIjogaW50KGZpbi5zdW0oKSksCiAgICAgICAgICAgICJsbG1faGVhZGxpbmUiOiBOb25lIGlmIG5vdCBucC5pc2Zpbml0ZShoZWFkbGluZSkgZWxzZSByb3VuZChmbG9hdChoZWFkbGluZSksIDQpLAogICAgICAgICAgICAibW9kZWwiOiBNT0RFTH0KCgpkZWYgbWFpbihuYW1lczogbGlzdFtzdHJdKSAtPiBOb25lOgogICAgY2ZnID0gbG9hZF9kYXRhX2NvbmZpZygpCiAgICBuYW1lcyA9IG5hbWVzIG9yIFtuIGZvciBuIGluIFJFR0lTVFJZIGlmIChjZmcucGF0aHMucHJvY2Vzc2VkIC8gbiAvICJlbmNvZGVyLnBhcnF1ZXQiKS5leGlzdHMoKV0KICAgIHRvaywgbW9kZWwgPSBsb2FkX2xsbSgpCiAgICBvdXQsIGV4YW1wbGVzID0ge30sIHt9CiAgICBmb3IgbiBpbiBuYW1lczoKICAgICAgICBwcmludChmIj09PSBMTE0gemVyby1zaG90OiB7bn0gKHtNT0RFTH0pID09PSIsIGZsdXNoPVRydWUpCiAgICAgICAgb3V0W25dID0gc2NvcmVfZGF0YXNldChuLCBjZmcsIHRvaywgbW9kZWwsIGV4YW1wbGVzKQogICAgICAgIHByaW50KGYiICAgIHtufToge291dFtuXX0iLCBmbHVzaD1UcnVlKQogICAgZCA9IGNmZy5wYXRocy5yZXBvcnRzIC8gInBoYXNlX2xsbSIKICAgIGQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgKGQgLyAibGxtX2Jhc2VsaW5lLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoCiAgICAgICAgeyJtb2RlbCI6IE1PREVMLCAibWF4X3Jvd3MiOiBNQVhfUk9XUywgInRlbXBlcmF0dXJlIjogMC4wLCAiZGF0YXNldHMiOiBvdXR9LAogICAgICAgIGluZGVudD0yLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAoZCAvICJleGFtcGxlcy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKGV4YW1wbGVzLCBpbmRlbnQ9MiwgZGVmYXVsdD1zdHIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgcHJpbnQoZiJ3cm90ZSB7ZCAvICdsbG1fYmFzZWxpbmUuanNvbid9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbihbYSBmb3IgYSBpbiBzeXMuYXJndlsxOl0gaWYgbm90IGEuc3RhcnRzd2l0aCgiLSIpXSkK'
)
# 11) Ensure the LLM baseline script is present. If your zip already has experiments/, we use it
#     as-is; otherwise we write the exact same script from an embedded (base64) copy - so this
#     notebook is self-sufficient regardless of how the zip was built.
import base64
dst = os.path.join(REPO, 'experiments', 'llm_zeroshot_baseline.py')
os.makedirs(os.path.dirname(dst), exist_ok=True)
if os.path.exists(dst):
    print('LLM baseline script already in the repo ->', dst)
else:
    open(dst, 'wb').write(base64.b64decode(LLM_BASELINE_B64))
    print('wrote LLM baseline script from embedded copy ->', dst)

In [ ]:
# 12) Run the zero-shot baseline (temperature 0). Capped at LLM_MAX_ROWS rows/dataset for cost;
#     raise it for full sets. Runs as a script (not -m) with PYTHONPATH already exported. Non-fatal.
env = dict(os.environ, LLM_MODEL='Qwen/Qwen2.5-3B-Instruct', LLM_MAX_ROWS='400', LLM_BATCH='16')
rc = subprocess.run([sys.executable, 'experiments/llm_zeroshot_baseline.py', *DATASETS],
                    cwd=REPO, env=env).returncode
print('LLM baseline exit code:', rc, '(non-zero is non-fatal - core results already saved)')
p = os.path.join(REPO, 'reports', 'phase_llm', 'llm_baseline.json')
if os.path.exists(p):
    import json
    print(json.dumps(json.load(open(p)), indent=2)[:2000])

In [ ]:
# 13) Bundle every new/updated artifact, download it, AND copy it to Drive (so results survive a
#     disconnect). Unzip results_bundle.zip over your local repo root to merge the new files.
OUTDIR = '/content/results_bundle'
shutil.rmtree(OUTDIR, ignore_errors=True); os.makedirs(OUTDIR)
patterns = ['data/processed/*/neural_oof.parquet',
            'reports/phase2g/results.*', 'reports/phase2g/preds/*',
            'reports/phase_hybrid/*', 'reports/phase_llm/*',
            'reports/phase3/*', 'reports/phase2d/*', 'reports/phase2f/*',
            'reports/figures/*.png']
n = 0
for pat in patterns:
    for f in glob.glob(os.path.join(REPO, pat), recursive=True):
        if os.path.isfile(f):
            rel = os.path.relpath(f, REPO); dstf = os.path.join(OUTDIR, rel)
            os.makedirs(os.path.dirname(dstf), exist_ok=True); shutil.copy2(f, dstf); n += 1
print(n, 'files bundled')
zip_path = shutil.make_archive('/content/results_bundle', 'zip', OUTDIR)
try:
    shutil.copy2(zip_path, '/content/drive/MyDrive/Explainable-Hybrid-ASAG-results.zip')
    print('copied results to MyDrive/Explainable-Hybrid-ASAG-results.zip')
except Exception as e:
    print('could not copy to Drive (non-fatal):', e)
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('browser download skipped (' + str(e) + ') - use the Drive copy above')

## After the run - fill the paper
1. Unzip `results_bundle.zip` (or the Drive copy) over your local repo root - it merges `neural_oof.parquet` + the new reports.
2. Key numbers now available:
   - `reports/phase_hybrid/three_way.json` -> Table 3 (`tab:hybrid`), RQ5 section, abstract sentence.
   - `reports/phase2g/results.json` -> neural-only column.
   - `reports/phase3/ablations.json` -> the `-D` / `only-D` rows.
   - `reports/phase2f/shap.json` -> share of global |SHAP| absorbed by `neural_*` (attribution dilution).
   - `reports/phase_llm/llm_baseline.json` -> the LLM baseline row + discussion.
3. Run `python paper/verify_numbers.py`, then replace each `\TBD{...}` / `TODO(after Colab run)` in `paper/main.tex`.

**Still open for a strong Q2 (from the runbook roadmap):** verify the ~12 flagged bib entries, add the runtime/memory table, LODO threshold-sensitivity + perturbation CIs, and a native-English editing pass.